# 🧠 DINOv2 Fine-Tuning — LEGO Piece Specialization

**Objetivo**: Especializar DINOv2-ViT-S14 en piezas LEGO usando Triplet Loss con Hard Negative Mining.

| Celda | Fase | Descripción |
|-------|------|-------------|
| C0 | Setup | Configuración del entorno e instalación de dependencias |
| C1 | Dataset | Descomprimir ZIP y preparar TripletDataset |
| C2 | Fine-Tune | Entrenar DINOv2 con Triplet Margin Loss |
| C3 | Export | Guardar pesos y generar ZIP de resultados |


In [ ]:
# =================================================================
# CELDA 0: Setup Lightning AI (DINOv2 Fine-Tuning)
# =================================================================
import os, sys, json, zipfile, glob, shutil, datetime

# --- Find PROJECT_ROOT ---
PROJECT_ROOT = ''
for parent in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(parent, 'src')):
        PROJECT_ROOT = parent
        break
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()

WORKSPACE_DIR = os.path.join(os.getcwd(), 'dinov2_workspace')
os.makedirs(WORKSPACE_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, WORKSPACE_DIR)

# --- Install dependencies ---
os.system('pip install torch torchvision faiss-cpu scikit-learn Pillow xformers -q')

# --- Detect GPU ---
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    device = torch.device('cpu')
    print('⚠️ No GPU detected. Training will be VERY slow.')

print(f'PyTorch: {torch.__version__} | Device: {device}')


In [ ]:
# =================================================================
# CELDA 1: 📦 Cargar Dataset y Preparar Triplets
# =================================================================
import os, zipfile, json, glob

print('>>> FASE 1: Carga del Dataset de Referencia')

# Find the DINOv2 dataset ZIP
zip_candidates = []
search_dirs = [PROJECT_ROOT, WORKSPACE_DIR, os.getcwd(),
               os.path.dirname(PROJECT_ROOT),
               os.path.dirname(os.getcwd())]
for search_dir in set(search_dirs):
    if os.path.isdir(search_dir):
        zip_candidates.extend(glob.glob(os.path.join(search_dir, 'lightning_dinov2_*.zip')))
        zip_candidates.extend(glob.glob(os.path.join(search_dir, 'kaggle_dinov2_*.zip')))

# Also check /kaggle/input/ for Kaggle datasets
for input_dir in glob.glob('/kaggle/input/*'):
    zip_candidates.extend(glob.glob(os.path.join(input_dir, '*dinov2*.zip')))

if not zip_candidates:
    raise FileNotFoundError('❌ No DINOv2 dataset ZIP found.')

dataset_zip = max(zip_candidates, key=os.path.getmtime)
print(f'📦 Dataset: {os.path.basename(dataset_zip)}')

# Extract
extract_dir = WORKSPACE_DIR
with zipfile.ZipFile(dataset_zip, 'r') as z:
    z.extractall(extract_dir)
    print(f'✅ Extracted to {extract_dir}')

# Locate ref_pieza and hard_negatives.json
RENDER_DIR = os.path.join(extract_dir, 'ref_pieza')
if not os.path.isdir(RENDER_DIR):
    # Try nested
    for d in os.listdir(extract_dir):
        cand = os.path.join(extract_dir, d, 'ref_pieza')
        if os.path.isdir(cand):
            RENDER_DIR = cand
            break

HN_PATH = os.path.join(extract_dir, 'hard_negatives.json')
if not os.path.exists(HN_PATH):
    HN_PATH = glob.glob(os.path.join(extract_dir, '**', 'hard_negatives.json'), recursive=True)
    HN_PATH = HN_PATH[0] if HN_PATH else None

if not os.path.isdir(RENDER_DIR):
    raise FileNotFoundError(f'❌ ref_pieza not found in {extract_dir}')
if not HN_PATH:
    raise FileNotFoundError('❌ hard_negatives.json not found')

with open(HN_PATH) as f:
    hn_data = json.load(f)
print(f'📊 Hard negatives: {hn_data["total_pieces"]} pieces, top-{hn_data["top_k"]} each')

# Count images
total_imgs = sum(
    len(os.listdir(os.path.join(RENDER_DIR, d, 'images')))
    for d in os.listdir(RENDER_DIR)
    if os.path.isdir(os.path.join(RENDER_DIR, d, 'images'))
)
print(f'🖼️ Total images: {total_imgs}')


In [ ]:
# =================================================================
# CELDA 2: 🏋️ Fine-Tune DINOv2 (Triplet Loss)
# =================================================================
import torch, torch.nn as nn, datetime, os
from torch.utils.data import DataLoader
from src.logic.triplet_dataset import TripletDataset

print('>>> FASE 2: Fine-Tuning DINOv2-ViT-S14')

# --- Hyperparameters ---
EPOCHS = 50
BATCH_SIZE = 128
LR = 1e-5
MARGIN = 0.3
UNFREEZE_LAYERS = 4
SAMPLES_PER_PIECE = 20
PATIENCE = 10

# --- Load Model ---
print('🧠 Loading DINOv2-ViT-S14...')
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

# Freeze all, then unfreeze last N blocks
for param in model.parameters():
    param.requires_grad = False

total_blocks = len(model.blocks)
for i in range(max(0, total_blocks - UNFREEZE_LAYERS), total_blocks):
    for param in model.blocks[i].parameters():
        param.requires_grad = True
if hasattr(model, 'norm'):
    for param in model.norm.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'📊 Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

model = model.to(device)
model.train()

# --- Dataset ---
dataset = TripletDataset(
    render_dir=RENDER_DIR,
    hard_negatives_path=HN_PATH,
    samples_per_piece=SAMPLES_PER_PIECE,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

# --- Optimizer & AMP ---
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

# --- Training Loop ---
log_path = os.path.join(WORKSPACE_DIR, 'training_log.csv')
with open(log_path, 'w') as f:
    f.write('epoch,loss,lr,timestamp\n')

best_loss = float('inf')
early_stop_counter = 0
output_dir = os.path.join(PROJECT_ROOT, 'models', 'dinov2_lego')
os.makedirs(output_dir, exist_ok=True)

for epoch in range(EPOCHS):
    total_loss, n_batches = 0.0, 0
    for anchors, positives, negatives in dataloader:
        a, p, n = anchors.to(device), positives.to(device), negatives.to(device)
        
        with torch.autocast(device_type=('cuda' if torch.cuda.is_available() else 'cpu'), dtype=torch.float16):
            a_emb = nn.functional.normalize(model(a), p=2, dim=1)
            p_emb = nn.functional.normalize(model(p), p=2, dim=1)
            n_emb = nn.functional.normalize(model(n), p=2, dim=1)
            loss = criterion(a_emb, p_emb, n_emb)
            
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        n_batches += 1
    scheduler.step()
    avg_loss = total_loss / max(n_batches, 1)
    lr_curr = scheduler.get_last_lr()[0]
    ts_log = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[EPOCH {epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f} | LR: {lr_curr:.2e}')
    
    # Write to log
    with open(log_path, 'a') as f:
        f.write(f'{epoch+1},{avg_loss:.6f},{lr_curr:.2e},{ts_log}\n')

    if avg_loss < best_loss - 0.0001:
        best_loss = avg_loss
        early_stop_counter = 0
        best_path = os.path.join(output_dir, 'dinov2_lego_best.pth')
        torch.save({'state_dict': model.state_dict(), 'epoch': epoch+1, 'loss': avg_loss,
                    'config': {'base_model': 'dinov2_vits14', 'unfreeze_layers': UNFREEZE_LAYERS,
                               'margin': MARGIN, 'feature_dim': 384}}, best_path)
        print(f'   ⭐ Best model updated: {best_path} (Loss: {avg_loss:.6f})')
    else:
        early_stop_counter += 1
        print(f'   ⚠️ No improvement ({early_stop_counter}/{PATIENCE})')
        if early_stop_counter >= PATIENCE:
            print(f'\n🛑 Early stopping triggered at epoch {epoch+1}')
            break

print(f'\n✅ Fine-tuning complete. Best loss: {best_loss:.4f}')


In [ ]:
# =================================================================
# CELDA 3: 📦 Export & ZIP Results
# =================================================================
import zipfile, datetime, os, glob

print('>>> FASE 3: Empaquetando Resultados')

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_name = f'dinov2_final_results_{ts}.zip'
zip_path = os.path.join(WORKSPACE_DIR, zip_name)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Include fine-tuned weights
    dinov2_dir = os.path.join(PROJECT_ROOT, 'models', 'dinov2_lego')
    if os.path.isdir(dinov2_dir):
        for f in os.listdir(dinov2_dir):
            if f.endswith('.pth'):
                fp = os.path.join(dinov2_dir, f)
                zipf.write(fp, arcname=f'models/dinov2_lego/{f}')
                print(f'   + {f}')

    # Include training log
    log_path = os.path.join(WORKSPACE_DIR, 'training_log.csv')
    if os.path.exists(log_path):
        zipf.write(log_path, arcname='training_log.csv')
        print('   + training_log.csv')

sz = os.path.getsize(zip_path) / (1024*1024)
print(f'✅ Results ZIP: {zip_path} ({sz:.1f} MB)')
print(f'📥 Download from the sidebar.')
